# 01 — Dataset Audit (Phase 1)

Executable driver for the Phase 1 audit. It runs the four audit scripts and
summarises their output. It contains **no modelling** and produces no predictions.

All logic lives in `src/data/` so the audit is reproducible from the command line:

```bash
py src/data/audit_datasets.py
py src/data/audit_geography.py
py src/data/target_feasibility.py
py src/data/resolve_denominator.py
```

Outputs are written to `datasets/metadata/` and `reports/dataset_report/`.

> **Note:** the numbers quoted in the cells below come from the recorded run in
> `Admin/EXPERIMENT_LOG.md`. Run the cell to regenerate them; outputs are not
> stored in this notebook file.

In [ ]:
from pathlib import Path
import subprocess, sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SCRIPTS = [
    'src/data/audit_datasets.py',
    'src/data/audit_geography.py',
    'src/data/target_feasibility.py',
    'src/data/resolve_denominator.py',
]

for script in SCRIPTS:
    print('=' * 70)
    print(script)
    proc = subprocess.run([sys.executable, str(ROOT / script)], capture_output=True, text=True)
    print(proc.stdout[-2500:])
    if proc.returncode != 0:
        print('STDERR:', proc.stderr[-2000:])

## 1. What was inspected

Ten files, all present, none downloaded. Row/column counts are read from the
generated manifest so they cannot drift from the data.

In [ ]:
import pandas as pd

META = ROOT / 'datasets' / 'metadata'
manifest = pd.read_csv(META / 'dataset_manifest.csv')
quality = pd.read_csv(META / 'data_quality_report.csv')

display(manifest[['dataset_id', 'n_rows', 'n_cols', 'geographic_level', 'time_period', 'role']])
print(quality['status'].value_counts().to_string())
display(quality[quality['status'] != 'PASS'])

## 2. Target feasibility — the decisive check

The proposed target was `Dilapidated_Households / Households`. The denominator is
not a household count:

- Σ `Households` = 330,835,767, which is India's **census-house stock** (330,983,625),
  not its 246,740,228 households.
- `Rural_Households + Urban_Households` equals the same 330,835,767.
- D1's asset columns *are* households (electricity 67.2%, TV 47.2%, latrine 46.9%,
  LPG 28.5% of 247M households, matching published Census 2011 shares).
- Per-district inflation ranges 0.18–2.63×, so the bias cannot be rescaled away.

The corrected target (Census-computed dilapidated share, D2) agrees with a
count-based reconstruction at **r = 0.9994, median difference 0.053 pp** (n = 628).

In [ ]:
crosswalk = pd.read_csv(META / 'district_crosswalk.csv')
print('districts:', len(crosswalk))
print('D2 matched:', int(crosswalk['d2_matched'].sum()))
print('reliable household denominator:', int(crosswalk['d5_match_reliable'].sum()))
display(
    crosswalk[['target_deprivation_rate', 'd2_dilapidated_pct', 'd2_minus_target_pp']]
    .describe()
    .round(3)
)

## 3. Verdict

| Check | Result |
| --- | --- |
| Numerator exists | Yes — 640 values, no nulls |
| Denominator exists | Yes, but it is the house stock, not households |
| Zero denominators | 0 |
| Missing targets silently zeroed | No |
| Compatible population | **No** — different universes |
| Geography reconciles | Yes — 640/640 after canonicalising state names |

**Proposed target rejected; corrected target validated and awaiting approval (D-001).**

See `Admin/DATASET_AUDIT.md` for the leakage assessment, quality findings, the
recommended dataset combination and the Phase 2 plan.